# EDA - Beijing Multi-Site Air Quality

Analisis exploratorio del **caso guia**: 12 estaciones de monitoreo en Beijing,
2013-03 a 2017-02, lecturas horarias de contaminantes y meteorologia.

- **Target** (regresion): `PM2.5` (ug/m3).
- **Particiones fijas**: train / valid / test / produccion.

Dos reglas que gobiernan este notebook:

1. **Las definiciones no se repiten aqui.** Columnas, particiones y contrato se
   importan de `BeijingAir`. Un EDA con su propia lista de features es la primera
   grieta por donde entra el train/serving skew.
2. **El analisis se hace sobre `train`.** Mirar `test` o `produccion` para decidir
   imputaciones, caps o features es leakage, aunque no lo parezca: la decision
   viaja del dato al modelo por la cabeza de quien programa.

## 1. Configuracion e imports

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from BeijingAir.config import COL_TIEMPO, PARTICIONES_PRODUCCION, PARTICIONES_TRAIN
from BeijingAir.data.contract import validar_crudos
from BeijingAir.data.descarga import cargar_crudo, filtrar
from BeijingAir.features.contract import TARGET

plt.rcParams["figure.figsize"] = (10, 5)
sns.set_theme(style="whitegrid")

CONTAMINANTES = ["PM2.5", "PM10", "SO2", "NO2", "CO", "O3"]
METEO = ["TEMP", "PRES", "DEWP", "RAIN", "WSPM"]

#: Los 16 rumbos en orden angular (N -> NNW). Ordenarlos alfabeticamente
#: destruiria la lectura: el viento es una variable circular.
ORDEN_RUMBOS = [
    "N", "NNE", "NE", "ENE", "E", "ESE", "SE", "SSE",
    "S", "SSW", "SW", "WSW", "W", "WNW", "NW", "NNW",
]

## 2. Carga del crudo

Se trabaja sobre el **crudo** (tal como llega del proveedor) para entender el dato:
nulos reales, rangos, tipos. El procesado ya trae decisiones nuestras encima.

In [ ]:
crudo = cargar_crudo()
print("Shape:", crudo.shape)
print("Rango temporal:", crudo[COL_TIEMPO].min(), "a", crudo[COL_TIEMPO].max())
print("\nTipos de dato:")
print(crudo.dtypes)
crudo.head()

## 3. El contrato de datos, ejecutado

El EDA es el lugar natural para ver el contrato corriendo sobre el lote real: si
`validar_crudos` pasa, todo lo que se describe mas abajo esta dentro de los
limites declarados en `data/contract.py`. Si algun dia falla, la excepcion dice
exactamente que regla se rompio, en vez de dejar una metrica degradandose en
silencio.

In [ ]:
validado = validar_crudos(crudo)
print("Filas validadas:", f"{len(validado):,}")
print("El crudo completo cumple RegistrosCrudos:", len(validado) == len(crudo))

## 4. Alcance: el analisis se hace sobre train

Las particiones son rangos de fecha fijos declarados en `config.py`. A partir de
aqui, `df` es **solo train**; `produccion` se reserva para la seccion 13, donde
se compara sin tomar ninguna decision de modelado sobre ella.

In [ ]:
df = filtrar(crudo, PARTICIONES_TRAIN[0])
produccion = filtrar(crudo, PARTICIONES_PRODUCCION[0])

for nombre, parte in (("train", df), ("produccion", produccion)):
    print(
        f"{nombre:<11} {len(parte):>8,} filas   "
        f"{parte[COL_TIEMPO].min()} a {parte[COL_TIEMPO].max()}"
    )

## 5. Eje temporal

Frecuencia horaria, un registro por estacion y hora. Las filas por mes muestran
la cobertura del tramo: 12 estaciones x 24 h x dias del mes.

In [ ]:
por_mes = df[COL_TIEMPO].dt.to_period("M").value_counts().sort_index()
por_mes.plot(kind="bar", figsize=(14, 4), title="Registros por mes (train)")
plt.ylabel("filas")
plt.show()

## 6. Cobertura por estacion

Las 12 estaciones aportan **exactamente** el mismo numero de horas (20.448 cada
una en train): el dataset esta perfectamente balanceado por estacion, porque las
12 se instrumentaron a la vez y con el mismo calendario. Lo que cambia entre ellas
no es el volumen, sino los **nulos** (seccion 7) y el **nivel de PM2.5**
(seccion 12).

In [ ]:
print("Estaciones:", df["station"].nunique())
df["station"].value_counts().plot(
    kind="bar", figsize=(10, 4), title="Registros por estacion (train)"
)
plt.ylabel("filas")
plt.show()

## 7. Nulos por columna

En este dataset los nulos **no son estructurales**: el sensor no reporto esa hora
(fallo de captura), no que la magnitud no exista. Esa distincion decide la
estrategia de imputacion en `data/loaders.py`.

In [ ]:
nulos = df.isna().sum()
tabla_nulos = pd.DataFrame({"nulos": nulos, "%": (df.isna().mean() * 100).round(2)})
tabla_nulos = tabla_nulos[tabla_nulos["nulos"] > 0].sort_values("nulos", ascending=False)
tabla_nulos

In [ ]:
tabla_nulos["nulos"].plot(kind="barh", figsize=(8, 5), title="Nulos por columna (train)")
plt.xlabel("nulos")
plt.show()

### 7.1 El nulo no es uniforme: patron por estacion

Agregado, cada contaminante falla un 1-5 % de las horas. Desagregado por estacion
la dispersion es mucho mayor: el sensor de CO va del **4,1 % al 13,7 %** de horas
perdidas segun la estacion. No es ruido repartido parejo, hay sensores
sistematicamente peores. Esto justifica el indicador `<col>_era_nulo` que conserva
`limpiar`: la ausencia misma lleva informacion, y borrarla imputando en silencio
la tira.

In [ ]:
cols_nulos = [*CONTAMINANTES, *METEO, "wd"]
nulos_estacion = df.set_index("station")[cols_nulos].isna().groupby(level=0).mean().mul(100)

plt.figure(figsize=(12, 6))
sns.heatmap(nulos_estacion, annot=True, fmt=".1f", cmap="rocket_r")
plt.title("% de nulos por estacion y columna (train)")
plt.xlabel("")
plt.ylabel("")
plt.show()

### 7.2 El nulo no es uniforme: patron en el tiempo

Los fallos se concentran en tramos, no se reparten parejo mes a mes: CO llega al
**28,5 %** de horas perdidas en su peor mes, frente a un ~5 % de media, y el peor
mes del conjunto (2014-02) pierde el 8,3 % de las lecturas de contaminantes.

Esto importa para el modelado: la imputacion por mediana de particion, razonable
sobre un 5 % disperso, se vuelve mucho mas dudosa cuando cubre casi un tercio de
un mes seguido. El flag `<col>_era_nulo` es lo que permite al modelo distinguir
ambos casos.

In [ ]:
mes = df[COL_TIEMPO].dt.to_period("M").astype(str)
nulos_mes = df.set_index(mes)[CONTAMINANTES].isna().groupby(level=0).mean().mul(100)

nulos_mes.plot(figsize=(14, 5), title="% de nulos por mes y contaminante (train)")
plt.ylabel("% nulos")
plt.xlabel("")
plt.xticks(rotation=90)
plt.legend(ncol=6)
plt.show()

## 8. Distribucion de contaminantes

Concentraciones muy sesgadas a la derecha; por eso los histogramas van en escala
logaritmica. Es seguro hacerlo aqui: **no hay ceros ni negativos** en ninguno de
los seis contaminantes, asi que la escala log no descarta filas en silencio.

In [ ]:
print("Valores <= 0 por contaminante (deben ser 0 para que log_scale sea seguro):")
print((df[CONTAMINANTES] <= 0).sum())

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.ravel(), CONTAMINANTES):
    sns.histplot(df[col].dropna(), bins=60, ax=ax, log_scale=True)
    ax.set_title(col)
plt.tight_layout()
plt.show()

## 9. Meteorologia

Variables meteorologicas en unidades SI (TEMP y DEWP en grados C, PRES en hPa,
RAIN en mm, WSPM en m/s). `RAIN` es casi siempre cero: la lluvia horaria es un
evento raro, no una variable continua bien poblada.

In [ ]:
fig, axes = plt.subplots(1, len(METEO), figsize=(20, 4))
for ax, col in zip(axes, METEO):
    sns.histplot(df[col].dropna(), bins=60, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

Y la regla fisica que el contrato tambien verifica: **el punto de rocio nunca
supera la temperatura**. Solo tiene sentido evaluarla donde ambas magnitudes
estan presentes.

In [ ]:
ambas = df.dropna(subset=["TEMP", "DEWP"])
print("Filas con TEMP y DEWP:", f"{len(ambas):,}")
print("Violaciones de TEMP < DEWP:", int((ambas["TEMP"] < ambas["DEWP"]).sum()))

## 10. Direccion del viento

`wd` es, junto con `station`, una de las dos categoricas nativas del dataset.
Se mira en orden angular, no alfabetico.

In [ ]:
conteo_wd = df["wd"].value_counts().reindex(ORDEN_RUMBOS)
conteo_wd.plot(kind="bar", figsize=(12, 4), title="Horas por rumbo de viento (train)")
plt.ylabel("filas")
plt.show()

El rumbo no es solo descriptivo: separa el target. El viento del **noroeste**
(NW/NNW/WNW) trae aire de las montanas y limpia la cuenca; el del **sureste**
(ESE/SE/E) llega de la llanura industrial y acumula. La diferencia entre el mejor
y el peor rumbo casi duplica la media de PM2.5, asi que `wd` merece entrar al
modelo como categorica y no descartarse por comodidad.

In [ ]:
pm_por_rumbo = df.groupby("wd")[TARGET].mean().reindex(ORDEN_RUMBOS)
pm_por_rumbo.plot(kind="bar", figsize=(12, 4), color="steelblue")
plt.axhline(df[TARGET].mean(), color="crimson", linestyle="--", label="media global")
plt.title("PM2.5 promedio por rumbo de viento (train)")
plt.ylabel("PM2.5 (ug/m3)")
plt.legend()
plt.show()

print(pm_por_rumbo.round(1))

## 11. Relaciones entre variables

Se incluye la meteorologia en la matriz, no solo los contaminantes: parte de la
senal aprovechable del modelo esta en como el viento y la humedad dispersan o
acumulan las particulas.

In [ ]:
corr = df[[*CONTAMINANTES, *METEO]].corr()
plt.figure(figsize=(11, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlacion entre contaminantes y meteorologia (train)")
plt.show()

print("Correlacion de cada variable con el target, de mayor a menor:")
print(corr[TARGET].drop(TARGET).sort_values(ascending=False).round(2))

Tres lecturas de esa matriz:

- **PM10 (~0.84) y CO (~0.75)** son los predictores mas fuertes: misma fuente de
  emision (combustion, trafico, polvo). No es causalidad, es co-ocurrencia.
- **O3 correlaciona en negativo** con casi todo, y muy fuerte con NO2 (~-0.5).
  Es quimica, no ruido: el NO consume ozono, y el ozono se forma con sol y calor,
  justo cuando la capa de mezcla dispersa las particulas.
- **WSPM en negativo (~-0.26)**: mas viento, menos particulas. Es el mecanismo de
  dispersion, y es lo que hace util a `wd` en la seccion 10.

In [ ]:
sns.scatterplot(
    data=df.dropna(subset=["PM10", TARGET]).sample(10_000, random_state=42),
    x="PM10",
    y=TARGET,
    alpha=0.3,
)
plt.title("PM2.5 vs PM10 (train, muestra de 10.000)")
plt.show()

## 12. El target PM2.5

Distribucion, extremos y los tres ciclos del dato: anual, diario y por estacion.

In [ ]:
print(df[TARGET].describe().round(1))
print("\nCuantiles altos:")
print(df[TARGET].quantile([0.5, 0.9, 0.99, 0.999, 1.0]).round(1))
print("\nFilas en el tope nominal del instrumento (999):", int((df[TARGET] == 999).sum()))

sns.histplot(df[TARGET].dropna(), bins=80)
plt.title("Distribucion de PM2.5 (train)")
plt.show()

**Sobre el valor 999.** En el dataset completo hay 1 fila con `PM2.5 == 999` y 3
con `PM10 == 999`: son valores extremos aislados, **no una saturacion masiva del
instrumento**. La cola pesada es real (el p99 esta en ~340 ug/m3 y el maximo en
~844), y esa es la dificultad del problema: el modelo tiene que acertar en
episodios de contaminacion severa que son raros en el entrenamiento. No hay
motivo para tratar 999 como censura ni para recortar por ese valor.

### 12.1 Ciclo anual

El ciclo anual es, con diferencia, **el mas fuerte de los tres**: entre el mes mas
limpio (agosto, ~62 ug/m3) y el mas sucio (febrero, ~120 ug/m3) hay ~58 ug/m3 de
diferencia, mas del triple que el ciclo horario. Los meses frios concentran las
medias altas, lo esperable en Beijing: calefaccion de carbon mas inversion
termica.

Cautela al leer la curva por mes calendario: train cubre 2,3 anios, asi que marzo
aparece tres veces y diciembre solo dos. Parte de la irregularidad entre meses es
ese desbalance, no estacionalidad. Aun asi la magnitud justifica `mes` y
`temporada` como features derivadas del calendario (disponibles antes de `t`, sin
leakage).

In [ ]:
con_target = df.dropna(subset=[TARGET])
serie_mes = con_target.groupby(con_target[COL_TIEMPO].dt.to_period("M"))[TARGET].mean()

serie_mes.plot(figsize=(14, 4), marker="o", title="PM2.5 promedio por mes (train)")
plt.ylabel("PM2.5 (ug/m3)")
plt.xlabel("")
plt.show()

### 12.2 Ciclo diario y semanal

El ciclo horario es claro: minimo a las 16 h (~75 ug/m3, capa de mezcla alta y
maxima dispersion) y maximo a las 22 h (~91 ug/m3).

El semanal sorprende. Con los dos ejes a la misma escala se ve que **su amplitud
es equivalente a la del ciclo diario** (~17 frente a ~16 ug/m3), y que va en
direccion contraria a la intuicion de "trafico de dias laborables": el minimo cae
en miercoles (~74) y el maximo en sabado (~91). No es un efecto de conmutacion
laboral; el fin de semana no limpia el aire de Beijing. `dia_semana` merece
quedarse como feature por derecho propio, no como relleno.

In [ ]:
hora = con_target[COL_TIEMPO].dt.hour
dia_semana = con_target[COL_TIEMPO].dt.dayofweek

fig, (izq, der) = plt.subplots(1, 2, figsize=(15, 4))
sns.lineplot(x=hora, y=con_target[TARGET], errorbar=None, ax=izq)
izq.set_title("PM2.5 promedio por hora")
izq.set_xlabel("hora")

sns.lineplot(x=dia_semana, y=con_target[TARGET], errorbar=None, ax=der)
der.set_title("PM2.5 promedio por dia de la semana (0=lunes)")
der.set_xlabel("dia de la semana")
der.set_ylim(izq.get_ylim())
plt.tight_layout()
plt.show()

### 12.3 Heterogeneidad por estacion

Las medianas por estacion difieren, pero mucho menos que la dispersion dentro de
cada una: el nivel base de la estacion aporta menos que la hora y el mes. El mapa
de calor muestra los dos ejes a la vez.

In [ ]:
sns.boxplot(data=con_target, x="station", y=TARGET)
plt.xticks(rotation=90)
plt.title("PM2.5 por estacion (train)")
plt.show()

In [ ]:
ciclo_estacion_hora = con_target.groupby([con_target["station"], hora])[TARGET].mean().unstack()

plt.figure(figsize=(14, 6))
sns.heatmap(ciclo_estacion_hora, cmap="mako")
plt.title("PM2.5 promedio por estacion y hora (train)")
plt.xlabel("Hora")
plt.ylabel("Estacion")
plt.show()

## 13. train vs produccion: que tan distinto es el futuro

Aqui **no se toma ninguna decision de modelado**; se mira para saber que esperar
del monitoreo. Es el equivalente manual de lo que despues vigila el chequeo de
drift sobre la particion de produccion simulada.

In [ ]:
comparacion = pd.concat(
    [df.assign(particion="train"), produccion.assign(particion="produccion")],
    ignore_index=True,
)
print(comparacion.groupby("particion")[TARGET].describe().round(1))

In [ ]:
fig, (izq, der) = plt.subplots(1, 2, figsize=(15, 4))
sns.kdeplot(
    data=comparacion, x=TARGET, hue="particion", log_scale=True, common_norm=False, ax=izq
)
izq.set_title("Distribucion de PM2.5 (escala log)")

sin_nulos = comparacion.dropna(subset=[TARGET])
sns.lineplot(
    x=sin_nulos[COL_TIEMPO].dt.hour,
    y=sin_nulos[TARGET],
    hue=sin_nulos["particion"],
    errorbar=None,
    ax=der,
)
der.set_title("Ciclo horario por particion")
der.set_xlabel("hora")
plt.tight_layout()
plt.show()

La media apenas se mueve (~82 vs ~83 ug/m3), pero **la mediana baja y la
desviacion sube**: produccion tiene menos horas de contaminacion media y mas
episodios extremos. Un monitor que solo vigile la media no veria nada; hay que
vigilar tambien la cola y el error por estacion, que es justo lo que reporta
`metricas_por_estacion.json`.

## 14. Conclusiones

1. **El contrato pasa** sobre las 420.768 filas: lo que sigue esta dentro de los
   limites declarados, no son observaciones sueltas.
2. **Nulos = fallo de captura** (1-5 % en contaminantes, 0,1 % en meteorologia) y
   **no son uniformes**: se concentran por estacion y por tramo temporal. De ahi
   el indicador `<col>_era_nulo` en vez de una imputacion muda.
3. **Sesgo fuerte a la derecha** en todos los contaminantes, sin ceros. El valor
   999 son 4 filas en todo el dataset: cola pesada real, no saturacion.
4. **Predictores mas fuertes del target**: PM10 (~0.84) y CO (~0.75); O3 aporta
   senal en negativo y WSPM captura la dispersion por viento.
5. **`wd` importa**: el rumbo del viento casi duplica la media de PM2.5 entre
   noroeste (limpio) y sureste (sucio). Entra al modelo como categorica.
6. **Ciclos**, por amplitud: anual (~58 ug/m3, pico en invierno) >> semanal
   (~17, pico en sabado) ~ diario (~16, minimo a las 16 h). El semanal no es
   plano y no sigue el patron laborable. Los tres se derivan del timestamp,
   sin leakage.
7. **train vs produccion**: media estable, cola mas pesada en produccion. El
   monitoreo debe mirar cuantiles altos y error por estacion, no solo la media.

**Implicaciones directas para el pipeline**: descartar filas sin target,
imputar por mediana conservando el flag de nulo, tratar `station` y `wd` como
categoricas, derivar `hora`/`dia_semana`/`mes`/`temporada` del timestamp, y
reportar MAE por estacion ademas del global.